# Geometric Algebra from Scratch, Verified Against kingdon

This notebook takes two fixed 3D vectors

$$a = 2e_1 + 3e_2 \qquad b = e_1 - e_2 + 4e_3$$

and computes their geometric, inner and outer products **twice**:

1. with plain Python only (no libraries), and
2. with the [`kingdon`](https://github.com/tBuLi/kingdon) geometric algebra library.

It then checks that both agree, and that the fundamental identity for vectors holds:

$$ab = a \cdot b + a \wedge b$$

We work in the 3D Euclidean algebra $Cl(3,0)$, which has $2^3 = 8$ basis blades:

$$1,\; e_1,\; e_2,\; e_3,\; e_{12},\; e_{13},\; e_{23},\; e_{123}$$

| Step | What | Why |
| :-- | :-- | :-- |
| 0 | Set up | Reproducibility |
| 1 | Basis vectors | Foundation of the algebra |
| 2 | Define $a$ and $b$ | The fixed inputs |
| 3 | Multivector representation | A product of vectors is not a vector |
| 4 | Geometric product | The core operation |
| 5 | Inner and outer products | Computed independently, from components |
| 6 | Verify $ab = a \cdot b + a \wedge b$ | The identity under test |
| 7 | Same operations in kingdon | The reference implementation |
| 8 | Automated comparison | Makes the claim testable |

## Step 0 — Set up

Steps 1–6 use no imports at all. `kingdon` is only needed from Step 7 onwards.

In [1]:
from itertools import product

import kingdon
from kingdon import Algebra

print("kingdon version:", kingdon.__version__)

kingdon version: 3.0.0


## Step 1 — Basis vectors

Three orthonormal basis vectors $e_1, e_2, e_3$, stored as plain lists:

$$
e_i \cdot e_j =
\begin{cases}
1 & i = j \\
0 & i \neq j
\end{cases}
$$

In [2]:
e1 = [1, 0, 0]
e2 = [0, 1, 0]
e3 = [0, 0, 1]

## Step 2 — Define a and b

Built from the basis with two small helpers, so every number is visible.

In [3]:
def scale(c, v):
    """Multiply a vector by a scalar."""
    return [c * x for x in v]

def add(v, w):
    """Add two vectors component by component."""
    return [x + y for x, y in zip(v, w)]

a = add(scale(2, e1), scale(3, e2))                    # 2e1 + 3e2
b = add(add(e1, scale(-1, e2)), scale(4, e3))          # e1 - e2 + 4e3

print("a =", a)
print("b =", b)

a = [2, 3, 0]
b = [1, -1, 4]


## Step 3 — Multivectors

The geometric product of two vectors is a **scalar plus a bivector**, so a 3-element list is no longer enough.
A multivector is stored as a dictionary with one coefficient per basis blade:

| Key | Grade | Meaning |
| :-- | --: | :-- |
| `'1'` | 0 | scalar |
| `'e1'`, `'e2'`, `'e3'` | 1 | vectors |
| `'e12'`, `'e13'`, `'e23'` | 2 | bivectors |
| `'e123'` | 3 | trivector |

In [4]:
BLADES = ['1', 'e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'e123']

def zero():
    """A multivector with every coefficient equal to 0."""
    return {blade: 0 for blade in BLADES}

def vec_to_mv(v):
    """Convert a plain vector [x, y, z] to a multivector."""
    M = zero()
    M['e1'], M['e2'], M['e3'] = v
    return M

def show(M):
    """Readable form of a multivector, e.g. '-1 - 5 e12 + 8 e13'."""
    terms = [f"{c:+g}" + ("" if blade == '1' else " " + blade)
             for blade, c in M.items() if c != 0]
    return " ".join(terms) if terms else "0"

A = vec_to_mv(a)
B = vec_to_mv(b)

print("A =", show(A))
print("B =", show(B))

A = +2 e1 +3 e2
B = +1 e1 -1 e2 +4 e3


## Step 4 — The geometric product

Everything follows from two rules for the basis vectors:

$$e_i e_i = 1 \qquad\qquad e_i e_j = -e_j e_i \quad (i \neq j)$$

**Step 4a — product of two basis blades.** To multiply, say, $e_{13}\,e_2$ we write out the indices `1 3 2`
and sort them. Each time an index jumps over a larger one, the second rule gives a factor of $-1$.
When an index meets a copy of itself, the first rule makes the pair disappear.

In [5]:
def indices(blade):
    """'e13' -> [1, 3];  '1' -> []"""
    return [] if blade == '1' else [int(c) for c in blade[1:]]

def blade_name(idx):
    """[1, 3] -> 'e13';  [] -> '1'"""
    return 'e' + ''.join(str(i) for i in idx) if idx else '1'

def blade_product(blade1, blade2):
    """Return (sign, blade) such that blade1 * blade2 = sign * blade."""
    idx = indices(blade1)
    sign = 1
    for x in indices(blade2):
        # x must jump over every index larger than itself: one sign flip each.
        sign *= (-1) ** sum(y > x for y in idx)
        if x in idx:
            idx.remove(x)          # e_i e_i = 1
        else:
            idx.append(x)
    return sign, blade_name(sorted(idx))

print("e1  * e1 =", blade_product('e1', 'e1'))
print("e1  * e2 =", blade_product('e1', 'e2'))
print("e2  * e1 =", blade_product('e2', 'e1'))
print("e13 * e2 =", blade_product('e13', 'e2'))

e1  * e1 = (1, '1')
e1  * e2 = (1, 'e12')
e2  * e1 = (-1, 'e12')
e13 * e2 = (-1, 'e123')


**Step 4b — product of two multivectors.** Distribute over every pair of blades and add up the results.

In [6]:
def geometric_product(M, N):
    """Geometric product of two multivectors."""
    result = zero()
    for blade1, c1 in M.items():
        for blade2, c2 in N.items():
            sign, blade = blade_product(blade1, blade2)
            result[blade] += sign * c1 * c2
    return result

ab = geometric_product(A, B)
print("ab =", show(ab))

ab = -1 -5 e12 +8 e13 +12 e23


**Check by hand.** Expanding $(2e_1 + 3e_2)(e_1 - e_2 + 4e_3)$ term by term:

| Term | Result |
| :-- | :-- |
| $(2e_1)(e_1)$ | $+2$ |
| $(2e_1)(-e_2)$ | $-2e_{12}$ |
| $(2e_1)(4e_3)$ | $+8e_{13}$ |
| $(3e_2)(e_1)$ | $-3e_{12}$ |
| $(3e_2)(-e_2)$ | $-3$ |
| $(3e_2)(4e_3)$ | $+12e_{23}$ |

Sum: $ab = -1 - 5e_{12} + 8e_{13} + 12e_{23}$, which matches the code.
The product of two vectors is a scalar plus a bivector.

## Step 5 — Inner and outer products

To make Step 6 a real test, these are computed **directly from the components**, without using the geometric product:

$$a \cdot b = \sum_i a_i b_i \qquad\qquad a \wedge b = \sum_{i<j} (a_i b_j - a_j b_i)\, e_{ij}$$

These formulas are for two **vectors**, which is the scope of this notebook.

In [7]:
def inner_product(v, w):
    """a . b for two vectors: a scalar."""
    result = zero()
    result['1'] = sum(x * y for x, y in zip(v, w))
    return result

def outer_product(v, w):
    """a ^ b for two vectors: a bivector."""
    result = zero()
    for i, j in [(0, 1), (0, 2), (1, 2)]:
        result[blade_name([i + 1, j + 1])] = v[i] * w[j] - v[j] * w[i]
    return result

dot_ab = inner_product(a, b)
wedge_ab = outer_product(a, b)

print("a . b =", show(dot_ab))
print("a ^ b =", show(wedge_ab))

a . b = -1
a ^ b = -5 e12 +8 e13 +12 e23


## Step 6 — Verify the identity

Two checks, both using only our own code:

1. $ab = a \cdot b + a \wedge b$
2. $a \cdot b = \tfrac{1}{2}(ab + ba)$ and $a \wedge b = \tfrac{1}{2}(ab - ba)$ — the symmetric and antisymmetric parts of the geometric product.

In [8]:
def mv_add(M, N):
    return {blade: M[blade] + N[blade] for blade in BLADES}

def mv_scale(c, M):
    return {blade: c * M[blade] for blade in BLADES}

ba = geometric_product(B, A)

print("ab            =", show(ab))
print("a . b + a ^ b =", show(mv_add(dot_ab, wedge_ab)))
print("(ab + ba) / 2 =", show(mv_scale(0.5, mv_add(ab, ba))))
print("(ab - ba) / 2 =", show(mv_scale(0.5, mv_add(ab, mv_scale(-1, ba)))))

assert ab == mv_add(dot_ab, wedge_ab)
assert dot_ab == mv_scale(0.5, mv_add(ab, ba))
assert wedge_ab == mv_scale(0.5, mv_add(ab, mv_scale(-1, ba)))
print("\nIdentity verified.")

ab            = -1 -5 e12 +8 e13 +12 e23
a . b + a ^ b = -1 -5 e12 +8 e13 +12 e23
(ab + ba) / 2 = -1
(ab - ba) / 2 = -5 e12 +8 e13 +12 e23

Identity verified.


## Step 7 — The same operations in kingdon

Same algebra, same vectors, using the library's operators: `*` geometric, `|` inner, `^` outer.

In [9]:
alg = Algebra(3, 0, 0)                       # Cl(3,0): three basis vectors squaring to +1

e1_k, e2_k, e3_k = alg.blades['e1'], alg.blades['e2'], alg.blades['e3']

a_k = 2 * e1_k + 3 * e2_k
b_k = e1_k - e2_k + 4 * e3_k

print("a     =", a_k)
print("b     =", b_k)
print("ab    =", a_k * b_k)
print("a . b =", a_k | b_k)
print("a ^ b =", a_k ^ b_k)

a     = 2 𝐞₁ + 3 𝐞₂
b     = 1 𝐞₁ + -1 𝐞₂ + 4 𝐞₃
ab    = -1 + -5 𝐞₁₂ + 8 𝐞₁₃ + 12 𝐞₂₃
a . b = -1
a ^ b = -5 𝐞₁₂ + 8 𝐞₁₃ + 12 𝐞₂₃


## Step 8 — Automated comparison

Reading the printed output is not a test. Here kingdon's results are converted to our dictionary format and compared with `assert`.

kingdon labels each blade with an integer bitmask rather than a name: bit 0 is $e_1$, bit 1 is $e_2$, bit 2 is $e_3$.
So `5` (binary `101`) is $e_{13}$ and `0` is the scalar.

In [10]:
def mask_to_blade(mask):
    """kingdon bitmask -> our blade name, e.g. 5 (0b101) -> 'e13'."""
    return blade_name([i + 1 for i in range(3) if mask >> i & 1])

def from_kingdon(kv):
    """Convert a kingdon multivector to our dictionary format."""
    result = zero()
    for mask, coeff in kv.items():
        result[mask_to_blade(mask)] = coeff
    return result

checks = {
    "ab   ": (ab,       a_k * b_k),
    "a . b": (dot_ab,   a_k | b_k),
    "a ^ b": (wedge_ab, a_k ^ b_k),
}

for label, (ours, theirs) in checks.items():
    assert ours == from_kingdon(theirs), f"{label} does not match kingdon"
    print(f"{label}: ours = {show(ours):32} kingdon = {show(from_kingdon(theirs)):32} match")

ab   : ours = -1 -5 e12 +8 e13 +12 e23         kingdon = -1 -5 e12 +8 e13 +12 e23         match
a . b: ours = -1                               kingdon = -1                               match
a ^ b: ours = -5 e12 +8 e13 +12 e23            kingdon = -5 e12 +8 e13 +12 e23            match


**Beyond the two vectors.** The result above only exercises a few entries of the multiplication table.
As a stronger check of `blade_product`, compare all 8 × 8 = 64 products of basis blades with kingdon.

In [11]:
mismatches = 0
for mask1, mask2 in product(range(8), repeat=2):
    sign, blade = blade_product(mask_to_blade(mask1), mask_to_blade(mask2))
    ours = zero()
    ours[blade] = sign
    theirs = alg.multivector({mask1: 1}) * alg.multivector({mask2: 1})
    mismatches += ours != from_kingdon(theirs)

assert mismatches == 0
print("All 64 basis-blade products match kingdon.")

All 64 basis-blade products match kingdon.


## Result

For $a = 2e_1 + 3e_2$ and $b = e_1 - e_2 + 4e_3$:

| Quantity | Value | Matches kingdon |
| :-- | :-- | :-: |
| $ab$ | $-1 - 5e_{12} + 8e_{13} + 12e_{23}$ | yes |
| $a \cdot b$ | $-1$ | yes |
| $a \wedge b$ | $-5e_{12} + 8e_{13} + 12e_{23}$ | yes |

The inputs are integers, so the agreement is exact, not approximate.